In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium, Flex a On-Prem (prostřednictvím IBM Quantum Platform API). Jsou ve stavu preview verze a mohou se měnit.

## Přehled
V kvantové chemii se problém elektronové struktury zaměřuje na hledání řešení elektronové Schrödingerovy rovnice – kvantových vlnových funkcí popisujících chování elektronů v systému. Tyto vlnové funkce jsou vektory komplexních amplitud, přičemž každá amplituda odpovídá příspěvku jedné možné konfigurace elektronů.

Základní stav je vlnová funkce systému s nejnižší energií a má zvláštní důležitost při studiu molekulárních systémů. Nejpřesnější přístup k výpočtu základního stavu uvažuje všechny možné konfigurace elektronů, ale pro větší systémy se stává nezvládnutelným, protože počet konfigurací roste exponenciálně s velikostí systému.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) je inovativní hybridní kvantově-klasická metoda pro přesný odhad základního stavu molekulárních systémů. Integruje kvantový hardware s klasickými počítači – kvantové procesory efektivně prozkoumávají kandidátní konfigurace elektronů a výsledná vlnová funkce se počítá na klasických počítačích. Díky generování kompaktních, ale chemicky přesných vlnových funkcí HI-VQE podporuje výzkum a objevy v kvantové chemii a vědě o materiálech.

![Obrázek znázorňující přehled algoritmu HI-VQE od Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE snižuje výpočetní složitost problému elektronové struktury tím, že efektivně odhaduje základní stav s vysokou přesností. Zaměřuje se na pečlivě vybranou podmnožinu nejrelevantnějších konfigurací elektronů, čímž optimalizuje jak přesnost, tak efektivitu.

Kombinací silných stránek klasických i kvantových počítačů HI-VQE iterativně zpřesňuje a vylepšuje aktuální odhad vlnové funkce. Jedinečné techniky konstrukce podprostoru pomáhají zefektivnit výběr konfigurací, takže uživatelé mají větší výpočetní kontrolu a lepší přesnost při simulacích kvantové chemie.

Pokud se chceš dozvědět více o algoritmu do hloubky, můžeš si [přečíst příslušný výzkumný článek](https://arxiv.org/abs/2503.06292).

## Popis
Počet konfigurací elektronů pro molekulární systém roste exponenciálně s velikostí systému. U určitých elektronových stavů, jako je základní stav, je však běžné, že pouze malá část konfigurací významně přispívá k energii stavu. Metody vybrané konfigurační interakce (SCI) využívají tuto řídkost ke snížení výpočetních nákladů tím, že identifikují a zaměřují se na nejrelevantnější konfigurace. Tato podmnožina konfigurací se označuje jako podprostor.

HI-VQE využívá přirozenou efektivitu kvantových počítačů při reprezentaci molekulárních systémů k usnadnění hledání podprostoru. Integruje klasické a kvantové podrutiny k řešení problému elektronové struktury s vysokou přesností. Na rozdíl od stávajících kvantových SCI metod HI-VQE kombinuje variační trénink, iterativní konstrukci podprostoru a předdiagonalizační filtrování konfigurací, čímž zvyšuje efektivitu snížením počtu kvantových měření, iterací a nákladů na klasickou diagonalizaci. HI-VQE lze proto aplikovat na větší molekulární systémy vyžadující více qubitů a snižuje náklady na řešení problému dané velikosti na stejnou úroveň přesnosti.

![Obrázek znázorňující podrobný popis každého kroku algoritmu HI-VQE od Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Pro výpočet základního stavu systému HI-VQE nejprve použije klasický chemický balíček PySCF k vygenerování molekulární reprezentace ze vstupů zadaných uživatelem, jako je geometrie molekuly a další molekulární informace. Poté vstoupí do hybridní kvantově-klasické optimalizační smyčky, která iterativně zpřesňuje podprostor tak, aby optimálně reprezentoval základní stav a zároveň minimalizoval počet zahrnutých konfigurací. Smyčka pokračuje, dokud nejsou splněna konvergenční kritéria, jako je velikost podprostoru nebo stabilita energie, načež jsou výstupem vypočítaná vlnová funkce základního stavu a energie. Tyto výsledky lze použít ke konstrukci přesných povrchů potenciální energie a k dalšímu rozboru systému.

Optimalizační smyčka se zaměřuje na úpravu parametrů kvantového Circuit tak, aby generoval vysoce kvalitní podprostor. HI-VQE nabízí tři možnosti kvantového Circuit: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) a [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Optimalizace je inicializována blízko referenčního stavu Hartree-Fock pro jeho obecnou vhodnost. Circuit je poté spuštěn na kvantovém zařízení a konfigurace jsou vzorkovány z výsledného kvantového stavu a vráceny jako binární řetězce. Kvůli šumu kvantového zařízení mohou být některé vzorkované konfigurace fyzikálně neplatné, pokud nesplňují zachování počtu elektronů nebo spinu. HI-VQE to řeší pomocí procesu obnovy konfigurací z balíčku [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), takže uživatelé mohou neplatné konfigurace buď opravit, nebo je vyřadit.

Platné konfigurace poté procházejí volitelným filtrovacím krokem, který odstraní ty, u nichž se předpokládá minimální příspěvek. Tím se snižuje dimenze podprostoru, a tedy i náklady na krok diagonalizace. Pokud je filtrování povoleno, vytvoří se předběžný subprostorový Hamiltonián z platných konfigurací a provede se diagonalizace s velmi volnými ukončovacími kritérii. Ačkoli je přesnost výsledných amplitud pro každou konfiguraci nízká, je efektivní pro předpovězení, které konfigurace v dané iteraci z podprostoru vyřadit, a je rychlá na výpočet.

Vybrané konfigurace jsou přidány do podprostoru a Hamiltonián systému je promítnut do tohoto podprostoru. Podprostor se iterativně aktualizuje a zachovává nejrelevantnější konfigurace napříč iteracemi. Tento přístup se liší od alternativních metod tím, že kvantový Circuit nemusí v každém kroku aproximovat celý základní stav.

Následně je subprostorový Hamiltonián klasicky diagonalizován, aby bylo získáno nejnižší vlastní číslo a odpovídající vlastní vektor, reprezentující aproximaci základního stavu a jeho energie. Jak se kvalita podprostoru prostřednictvím iterací zlepšuje, vypočítaný základní stav lépe aproximuje skutečný základní stav. V tomto bodě lze provést dodatečný filtrovací krok k odstranění konfigurací z podprostoru, které významně nepřispívají k vypočítanému základnímu stavu. Tento krok zajišťuje, že podprostor přenesený do další iterace je co nejkompaktnější. Je vyhodnocován na základě amplitud vrácených diagonalizací, protože ty reprezentují příspěvek každé konfigurace k vypočítanému základnímu stavu.

Kontrola konvergence pak určí, zda by další trénink zlepšil výsledky. Pokud ano, provede se volitelný krok klasické expanze, parametry kvantového Circuit se aktualizují pro další minimalizaci vypočítané energie a proces se opakuje. Krok klasické expanze generuje další konfigurace pro podprostor, doplňující konfigurace vzorkované z kvantového zařízení. Nejprve identifikuje konfiguraci s největší amplitudou ve výsledcích diagonalizace a poté generuje nové konfigurace s jednoduchými a dvojnásobnými excitacemi z identifikované konfigurace. Požadovaný počet těchto konfigurací je poté přidán do podprostoru.

Jakmile je určeno, že iterace konvergovaly, HI-VQE vrátí vypočítaný základní stav (ve formě stavů v podprostoru a jejich amplitud ve vlnové funkci základního stavu), jeho energii a míru rozptylu energie, která ukazuje, zda vypočítaný stav tvoří vlastní stav Hamiltoniánu systému.

Uživatelé mohou rozhodovat o tom, který kvantový Circuit se použije, a o počtu shotů pro každý kvantový Circuit, a také řídit velikost podprostoru nebo povolit klasické generování dalších konfigurací na podporu konfigurací generovaných kvantově. Tímto způsobem mohou uživatelé přizpůsobit chování HI-VQE svým požadovaným aplikacím.

## Začínáme
Nejprve si [požádej o přístup k funkci](https://forms.office.com/r/zN3hvMTqJ1).
Poté se autentizuj pomocí svého [IBM Quantum&reg; API klíče](http://quantum.cloud.ibm.com/) a za předpokladu, že jsi již [uložil/a svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do lokálního prostředí, vyber Qiskit Function takto:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Vstupy
Všechny vstupní parametry, které funkce přijímá, jsou uvedeny v následující tabulce. Podrobnosti o příslušných argumentech najdeš v následujících sekcích [Možnosti molekuly](#molecule-options) a [Možnosti HI-VQE](#hi-vqe-options).
| Název                  | Typ                                                            | Popis                                                                                                                                                                                                                                                                                                       | Povinný  | Výchozí                                                                  | Příklad                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Může jít o řetězec nebo strukturované seznamy obsahující dvojice atomů a souřadnic. Pokud je zadán jako řetězec, musí obsahovat geometrii molekuly ve formátu kartézských souřadnic. Pokud je zadán jako seznam, musí jít o seznam seznamů, z nichž každý obsahuje řetězec atomu a n-tici souřadnic. | Ano      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` nebo `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Název backendu, na který se dotaz odešle.                                                                                                                                                                                                                                                                   | Ano      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Maximální dimenze podprostoru pro diagonalizaci. Pokud zadané číslo není dokonalý čtverec, použije se méně stavů.                                                                                                                                                                                           | Ano      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Maximální počet klasicky generovaných CI stavů zahrnutých do každé iterace.                                                                                                                                                                                                                                 | Ano      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Možnosti související s molekulou použitou jako vstup do HI-VQE. Více informací najdeš v sekci [Možnosti molekuly](#molecule-options).                                                                                                                                                                       | Ne       | Viz sekce [Možnosti molekuly](#molecule-options).                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Možnosti řídící chování algoritmu HI-VQE. Více informací najdeš v sekci [Možnosti HI-VQE](#hi-vqe-options).                                                                                                                                                                                                | Ne       | Viz sekce [Možnosti HI-VQE](#hi-vqe-options).                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### Možnosti molekuly
Následující tabulka podrobně popisuje všechny klíče a hodnoty, které lze nastavit ve slovníku `molecule_options`, včetně jejich datových typů a výchozích hodnot. Všechny klíče jsou volitelné.

| Klíč                              | Typ hodnoty                         | Výchozí hodnota                  | Platný rozsah                                                                                                                                                  | Vysvětlení                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Různé                                                                                                                                                          | Celé číslo udávající celkový čistý náboj molekulárního systému. Výchozí hodnota je 0, ale může to být libovolné celé číslo.                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Různé                                                                                                                                                          | Řetězec určující typ báze; tyto hodnoty se předávají pyscf. (např.: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Každý index orbitalu.            | Indexy prostorových orbitalů platné pro daný problém.                                                                                                          | Seznam indexů aktivních orbitalů v intervalu [0, n), kde n je počet qubitů použitých v problému. Pokud je tento parametr zadán, musí být zadán také argument frozen_orbitals.                                                                                                                                                                                                                                                                                                                                                           |
| `"frozen_orbitals"`               | `List[int]`                         | Žádné indexy.                    | Indexy prostorových orbitalů platné pro daný problém, kromě aktivních orbitalů.                                                                                | Seznam indexů zmrazených orbitalů ve stejném rozsahu jako active_orbitals. Pokud je zadán, musí být zadán také active_orbitals. Zmrazovat by se měly pouze obsazené orbitaly, protože počet aktivních elektronů se snižuje o 2 pro každý zmrazený obsazený orbital.                                                                                                                                                                                                                                                                    |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Molekulární orbitaly Hartree-Fock. | Různé.                                                                                                                                                        | Koeficienty prostorových orbitalů použité při výpočtu integrálů elektronické repulze pro systém. Platnými příklady jsou molekulární orbitaly Hartree-Fock, přirozené orbitaly a orbitaly AVAS.                                                                                                                                                                                                                                                                                                                                          |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` nebo `False`                                                                                                                                            | Slouží k aktivaci symetrie bodové grupy pro úvodní molekulární výpočty za účelem sestavení báze orbitálů adaptované na symetrii. Tyto orbitaly adaptované na symetrii se používají jako bázové funkce pro následné výpočty SCF. Výchozí hodnota je False; pokud je nastavena na True, symetrie se aktivuje a bodové grupy se detekují a použijí automaticky. Pokud je přiřazena konkrétní symetrie, např. symmetry = "Dooh", vyvolá se chyba, pokud molekulární geometrie dané symetrii neodpovídá. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Viz [dokumentace pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Lze použít ke generování podgrupy detekované symetrie. Nemá žádný účinek, pokud je symetrie zadána pomocí klíčového argumentu symmetry.                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Viz [dokumentace pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Určuje měrnou jednotku pro atomové souřadnice a vzdálenosti. Výchozí nastavení používá jednotky angstrom.                                                                                                                                                                                                                                                                                                                                                                                                                               |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Viz [dokumentace pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Určuje jaderný model, který se má použít. Výchozí je bodový jaderný model; jiné hodnoty aktivují Gaussovský jaderný model. Pokud je zadána funkce, použije se s Gaussovským jaderným modelem k vygenerování hodnoty rozložení jaderného náboje „zeta".                                                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Viz [dokumentace pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Určuje pseudopotenciál pro atomy v molekule. Výchozí hodnota je None, což znamená, že se nepoužívají žádné pseudopotenciály a všechny elektrony jsou explicitně zahrnuty do výpočtů.                                                                                                                                                                                                                                                                                                                                                              |
| `"cart"`                          | `bool`                              | `False`                          | Viz [dokumentace pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Určuje, zda se mají jako bázové funkce pro moment hybnosti ve výpočtu použít kartézské GTO. Výchozí hodnota False bude používat sférické GTO.                                                                                                                                                                                                                                                                                                                                                                                          |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Viz [dokumentace pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Nastavuje kolineární spinový magnetický moment každého atomu. Ve výchozím nastavení je None a každý atom je inicializován se spinem nula.                                                                                                                                                                                                                                                                                                                                                                                               |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | např. ["H 1s", "O 2p"] pro H2O                                                                                                                                | Definuje atomové orbitaly, které mají být zahrnuty do schématu AVAS. Viz [dokumentace AVAS](https://arxiv.org/abs/1701.07862) .                                                                                                                                                                                                                                                                                                                                                                                                          |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Mezi 0,0 a 2,0                                                                                                                                                 | Určuje mezní hodnotu používanou k rozhodnutí, které atomové orbitaly (AO) jsou zachovány v aktivním prostoru.                                                                                                                                                                                                                                                                                                                                                                                                                            |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` nebo `"ccsd"`                                                                                                                                          | Definuje teoretický přístup k přípravě přirozených orbitalů a výběru aktivních orbitalů na základě schématu přirozených čísel obsazení orbitalů (NOONs). Viz [dokumentace NOONs](https://doi.org/10.1063/5.0042147). Pro řízení počtu orbitalů (a počtu qubitů) musí být zadány jak indexy aktivních, tak zmrazených orbitalů.                                                                                                                                                                                                           |
### Možnosti HI-VQE
Následující tabulka popisuje všechny klíče a hodnoty, které lze nastavit ve slovníku `hivqe_options`, spolu s jejich datovými typy a výchozími hodnotami. Všechny klíče jsou volitelné.

| Klíč                              | Typ hodnoty                         | Výchozí hodnota                  | Platný rozsah                                                                                                                                                  | Vysvětlení                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Mezi 1 a 10 000.                                                                                                                                               | Počet měření na kvantovém zařízení za každou iteraci.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"max_iter"`                      | `int`                               | `25`                             | Mezi 1 a 50.                                                                                                                                                   | Maximální počet iterací pro optimalizaci ansatzu. Algoritmus může skončit dříve, pokud dojde k předčasné konvergenci.                                                                                                                                                                                                                                                                                                                                                                                                                    |
| `"initial_basis_states"`          | `List[str]`                         | Stav Hartree-Fock.               | Binární řetězce, jejichž počet bitů odpovídá požadovanému počtu qubitů pro daný problém.                                                                       | Lze použít k restartování algoritmu s klasickými stavy z předchozího výsledku.                                                                                                                                                                                                                                                                                                                                                                                                                                                          |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` nebo `"lucj"`                                                                                                                                 | Určuje kvantový ansatz, který se optimalizuje za účelem generování nových stavů. `"epa"` vybere ansatz zachovávající excitace. `"hea"` vybere hardwarově efektivní ansatz. `"lucj"` vybere ansatz lokálního unitárního klastrového Jastrowova přístupu.                                                                                                                                                                                                                                                                                  |
| `"convergence_count"`             | `int`                               | `3`                              | Alespoň 2.                                                                                                                                                     | Počet iterací bez výrazné změny vypočítané energie, které musí proběhnout, než je algoritmus považován za konvergovaný.                                                                                                                                                                                                                                                                                                                                                                                                                  |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Větší než 0 a nejvýše 1.                                                                                                                                       | Velikost změny vypočítané energie, která je považována za výraznou pro účely kontroly konvergence.                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` nebo `False`                                                                                                                                            | Pokud je `True`, musí proběhnout `convergence_count` iterací bez přerušující výrazné změny, aby byl algoritmus vyhodnocen jako konvergující. Pokud je `False`, algoritmus se zastaví po `convergence_count` iteracích s nevýraznými změnami, které se vyskytly kdykoliv v průběhu optimalizace.                                                                                                                                                                                                                                           |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` nebo `False`.                                                                                                                                           | Určuje, zda se má použít obnovení konfigurace z balíčku `qiskit-addon-sqd`. Pokud je True, neplatné stavy vzorkované z kvantového zařízení jsou klasicky opraveny. Pokud je False, jsou zahozeny.                                                                                                                                                                                                                                                                                                                                       |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Libovolná z hodnot `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` nebo `"sca"`. Při použití ansatzu `"lucj"` je dostupná i možnost `"lucj_default"`. | Určuje schéma provázání, které má být použito v rámci kvantového obvodu, v souladu s běžnými konvencemi Qiskit a [konvencemi ffsim pro ansatz LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                                                        |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Větší než 0.                                                                                                                                                   | Počet opakování každé vrstvy v kvantovém obvodu.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Alespoň 0 a méně než 1.                                                                                                                                        | Tolerance pro rozhodování, které stavy mají být po diagonalizaci vyřazeny z podprostoru. Určuje práh zahrnutí stavů podprostoru na základě jejich vypočítaných amplitud.                                                                                                                                                                                                                                                                                                                                                               |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Mezi `1e-4` a `1e-1` včetně.                                                                                                                                  | Tolerance pro předpovídání, které stavy mají být před diagonalizací vyřazeny z podprostoru. Řídí přesnost předpovídaných amplitud pro každý stav; nižší hodnota vede k přesnějším předpovědím.                                                                                                                                                                                                                                                                                                                                         |
## Výstupy
Funkce vrací slovník se čtyřmi klíči a hodnotami. Klíče a hodnoty jsou zdokumentovány v následující tabulce:

| Klíč                | Typ hodnoty   | Vysvětlení                                                                                                                |
|:--------------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`          | `float`       | Přibližná energie základního stavu molekuly.                                                                              |
| `"states"`          | `List[str]`   | Vybrané determinanty tvořící podprostor použitý k výpočtu energie. Jsou ve střídavém formátu alfa-beta.                   |
| `"eigenvector"`     | `List[float]` | Vlastní vektor odpovídající základnímu stavu podprostoru složeného ze stavů `"states"`.                                   |
| `"energy_variance"` | `float`       | Energetický rozptyl základního stavu podprostoru složeného ze stavů `"states"`, který udává kvalitu řešení. Tato hodnota je nezáporná a nižší hodnota znamená, že základní stav podprostoru lépe aproximuje vlastní stav Hamiltoniánu systému. |
| `"energy_history"`  | `List[float]` | Energie vypočítané v každé iteraci během hybridního optimalizačního procesu, ve stejném pořadí, v jakém byly vypočítány. V rámci optimalizačního procesu SPSA jsou v každé iteraci vypočítány dvě energie. |
## Příklad
První příklad ukazuje, jak vypočítat energii základního stavu molekuly NH3 pomocí algoritmu HI-VQE.
#### Definice molekulární geometrie a možností
Molekulární geometrie NH3 je zadána v kartézských souřadnicích, kde jsou atomy odděleny znakem „;".

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Další možnosti pro molekulární systém lze definovat a zadat v následujícím formátu slovníku.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Spusť funkci se zadanými vstupy geometrie a možností.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Je dobré vytisknout ID úlohy funkce, aby ho bylo možné uvést v žádostech o podporu v případě problémů.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Tento příklad využívá 16 qubitů s 8 orbitaly báze sto3g pro molekulu NH3.
Zkontroluj [stav](/guides/functions#check-job-status) svého pracovního zatížení Qiskit Function nebo načti [výsledky](/guides/functions#retrieve-results) takto:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Po dokončení úlohy lze výsledky získat pomocí instance `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Pro přístup k energii základního stavu použij klíč `"energy"`. Klíč `"eigenvector"` poskytuje CI koeficienty s odpovídající zápisem bitového řetězce konfigurace elektronů uloženého v `"states"` výsledků.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Výstup:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Výkon
Tato sekce ukazuje provedené referenční výpočty HI-VQE pro případ s 24 qubity pro Li2S, případ s 40 qubity pro molekulu N2 a případ s 44 qubity pro systém FeP-NO.
#### Křivka potenciální energetické plochy disociace molekuly Li2S s 24 qubity
Křivka PES je zobrazena s referenčními hodnotami FCI a počátečním odhadem z RHF spolu s energetickou chybou oproti referenční hodnotě FCI.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Výpočty byly provedeny s následujícími geometriemi a možnostmi.